# P4 · Улучшение CountLoader: качество самой памяти `M`

**Цель.** CountLoader ранжирует соседей по плотной per-pair памяти `M[u,i](t) = Σ φ(w_e)·exp(−α(t−t_e))` (additive-EMA, rank/ECDF, κ≈40 → standalone NDCG@10 **0.522** на genre). p4-диагностика показала: потолок внутри support = **0.987**, т.е. нужные item'ы в строке ЕСТЬ, но **неверно упорядочены**. Здесь ищем лучшее **одно состояние + одно правило записи/чтения** (не ансамбль!), чтобы сам ранкер выдавал больше.

**Рамки:** строго-каузальный стрим-eval (как p3/p4), итерации на genre (~15–30 с/конфиг), подтверждение победителя на reddit. Санкционировано backlog'ом (T7/H5): delta-rule / erase-then-add, GLA-style per-row decay, нормализации чтения. Запрещено: смешивать скоры двух матриц/предикторов (popularity-backoff уже измерен — мёртв).

**План:** (1) базлайн-анкер; (2) диагноз мис-ранжирования (сатурация тёплых строк? staleness? доминирование популярных колонок?); (3) эксперименты с правилами — дешёвые сначала; (4) подтверждение победителя на reddit; (5) вердикт → что менять в `models/msampler.py`.

In [1]:
# Setup — обобщённая per-pair память с плагируемыми правилами + строго-каузальный eval (харнес p3/p4)
import numpy as np, polars as pl, plotly.express as px, plotly.graph_objects as go, sys
from sklearn.metrics import ndcg_score
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader

def load_dataset(name, bs=200):
    ds = PyGNodePropPredDataset(name=name, root="datasets")
    data = ds.get_TemporalData()
    tr, va, te = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    return dict(ds=ds, data=data, num_classes=ds.num_classes, num_nodes=data.num_nodes,
                loaders={s: TemporalDataLoader(d, batch_size=bs) for s, d in [("train", tr), ("val", va), ("test", te)]})

def make_rank_phi(DS):  # rank/ECDF по train (каузально)
    tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
    sw = np.sort(tr.msg[:, 0].numpy()); n = len(sw)
    return lambda w: np.searchsorted(sw, w, side="right") / n

class PairMemory:
    """Одно состояние M[N,C] + одно правило. write: 'add' (базлайн) | 'ema' (delta-rule, beta).
    alpha — скаляр или per-user вектор [N]. read_tf(rows, users, t_read) — хук нормализации чтения."""
    def __init__(self, N, C, alpha, phi, write="add", beta=0.1, read_tf=None):
        self.C = C; self.M = np.zeros((N, C)); self.T = np.zeros((N, C))
        self.alpha = np.broadcast_to(np.asarray(alpha, dtype=np.float64), (N,)).copy()
        self.phi = phi; self.write = write; self.beta = float(beta); self.read_tf = read_tf
    def update(self, src, dst, t, w):
        if src.size == 0: return
        s64 = src.astype(np.int64); idx = s64 * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True)
        sphi = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)      # сумма phi по ячейке
        k = np.bincount(inv, minlength=uniq.size)                               # событий на ячейку
        au = self.alpha[(uniq // self.C)]                                       # per-user alpha ячеек
        fM, fT = self.M.reshape(-1), self.T.reshape(-1)
        dec = np.exp(-au * np.clip(tb - fT[uniq], 0, None))
        if self.write == "add":
            fM[uniq] = fM[uniq] * dec + sphi
        else:  # 'ema' delta-rule: m <- m*(1-b)^k + (1-(1-b)^k)*mean(phi)  (батч-гранулярность)
            g = (1 - self.beta) ** k
            fM[uniq] = fM[uniq] * dec * g + (1 - g) * (sphi / np.maximum(k, 1))
        fT[uniq] = tb
    def read(self, users, t_read):
        r = self.M[users] * np.exp(-self.alpha[users, None] * np.clip(t_read - self.T[users], 0, None))
        return self.read_tf(r, users, t_read) if self.read_tf else r

def eval_ranker(DS, mem, quiet=True):
    """Строго-каузальный stream-eval: train+val прогрев, test — NDCG@10 по (user,день)."""
    ds = DS["ds"]; ndcgs = []
    def stream(loader, collect):
        label_t = ds.get_label_time()
        for b in loader:
            s, d, t = b.src.numpy(), b.dst.numpy(), b.t.numpy(); w = b.msg[:, 0].numpy()
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
                pm = t < l0; mem.update(s[pm], d[pm], t[pm], w[pm])
                if collect:
                    pr = mem.read(us, l0)
                    for i in range(len(us)):
                        if ys[i].sum() > 0: ndcgs.append(ndcg_score(ys[i:i+1], pr[i:i+1], k=10))
                s, d, t, w = s[~pm], d[~pm], t[~pm], w[~pm]
            mem.update(s, d, t, w)
    stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], False)
    stream(DS["loaders"]["test"], True); ds.reset_label_time()
    return float(np.mean(ndcgs))

DAY = 86400.0
DS = load_dataset("tgbn-genre"); rphi = make_rank_phi(DS)
N, C = DS["num_nodes"], DS["num_classes"]
print(f"genre: N={N} C={C} edges={DS['data'].src.numel():,} — харнес готов")

genre: N=1505 C=513 edges=17,858,395 — харнес готов


In [2]:
# Базлайн-анкер: PairMemory(write='add') должна воспроизвести прежние числа (κ=25→0.5203, κ=40→0.5219)
import time
res = {}
for kappa in [25, 40]:
    t0 = time.time()
    nd = eval_ranker(DS, PairMemory(N, C, np.log(2)/(kappa*DAY), rphi, write="add"))
    res[kappa] = nd
    print(f"κ={kappa:>3}: NDCG@10 = {nd:.4f}   ({time.time()-t0:.0f} c)")
assert abs(res[25] - 0.5203) < 1e-3 and abs(res[40] - 0.5219) < 1e-3, "паритет с p4-числами нарушен!"
print("ПАРИТЕТ ✓ — обобщённый класс эквивалентен базлайну; далее все правила сравниваем с 0.5219 (κ=40)")

κ= 25: NDCG@10 = 0.5203   (14 c)


κ= 40: NDCG@10 = 0.5219   (14 c)
ПАРИТЕТ ✓ — обобщённый класс эквивалентен базлайну; далее все правила сравниваем с 0.5219 (κ=40)


In [3]:
# Диагноз мис-ранжирования (κ=40): сравниваем FP (в топ-10 M, но не позитив) vs FN (позитив, но вне топ-10)
# по трём осям: свежесть ячейки, её масса, глобальная популярность колонки.
class DiagMemory(PairMemory):
    def __init__(self, *a, **k):
        super().__init__(*a, **k); self.cnt = np.zeros_like(self.M)     # per-cell счётчик событий
        self.colpop = np.zeros(self.C)                                   # глобальная популярность колонок
    def update(self, src, dst, t, w):
        super().update(src, dst, t, w)
        if src.size: np.add.at(self.cnt, (src.astype(np.int64), dst.astype(np.int64)), 1); np.add.at(self.colpop, dst.astype(np.int64), 1)

mem = DiagMemory(N, C, np.log(2)/(40*DAY), rphi, write="add")
ds = DS["ds"]; rows = []
def stream_diag(loader, collect):
    label_t = ds.get_label_time()
    for b in loader:
        s, d, t = b.src.numpy(), b.dst.numpy(), b.t.numpy(); w = b.msg[:, 0].numpy()
        if float(b.t[-1]) > label_t:
            lt = ds.get_node_label(b.t[-1])
            if lt is None: break
            l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
            pm = t < l0; mem.update(s[pm], d[pm], t[pm], w[pm])
            if collect:
                pr = mem.read(us, l0); poprank = np.argsort(np.argsort(-mem.colpop))  # 0 = самый популярный
                for i in range(len(us)):
                    yi = ys[i]
                    if yi.sum() <= 0: continue
                    u = int(us[i]); top10 = np.argsort(-pr[i])[:10]; pos = np.where(yi > 0)[0]
                    seen = mem.cnt[u] > 0
                    fn = np.setdiff1d(pos[seen[pos]], top10)             # видённые позитивы вне топ-10
                    fp = np.setdiff1d(top10, pos)                        # топ-10 без метки сегодня
                    for grp, items in (("FN (упущенный позитив)", fn), ("FP (лишний в топ-10)", fp)):
                        for it in items[:20]:
                            rows.append((grp, (l0 - mem.T[u, it]) / DAY, mem.cnt[u, it], int(poprank[it])))
            s, d, t, w = s[~pm], d[~pm], t[~pm], w[~pm]
        mem.update(s, d, t, w)
stream_diag(DS["loaders"]["train"], False); stream_diag(DS["loaders"]["val"], False)
stream_diag(DS["loaders"]["test"], True); ds.reset_label_time()
df = pl.DataFrame(rows, schema=["группа", "staleness_дней", "событий_в_ячейке", "поп_ранг_колонки"], orient="row")
print(df.group_by("группа").agg(pl.len().alias("n"),
      pl.col("staleness_дней").median().round(1).alias("staleness_мед"),
      pl.col("событий_в_ячейке").median().alias("событий_мед"),
      pl.col("поп_ранг_колонки").median().alias("поп_ранг_мед")).sort("группа"))

shape: (2, 5)
┌────────────────────────┬────────┬───────────────┬─────────────┬──────────────┐
│ группа                 ┆ n      ┆ staleness_мед ┆ событий_мед ┆ поп_ранг_мед │
│ ---                    ┆ ---    ┆ ---           ┆ ---         ┆ ---          │
│ str                    ┆ u32    ┆ f64           ┆ f64         ┆ f64          │
╞════════════════════════╪════════╪═══════════════╪═════════════╪══════════════╡
│ FN (упущенный позитив) ┆ 201883 ┆ 5.2           ┆ 115.0       ┆ 46.0         │
│ FP (лишний в топ-10)   ┆ 193887 ┆ 2.3           ┆ 570.0       ┆ 16.0         │
└────────────────────────┴────────┴───────────────┴─────────────┴──────────────┘


**Диагноз (κ=40, ~200k пар FP/FN).** Ошибочный топ-10 состоит из ячеек: **тяжёлых** (медиана **570** событий против **115** у упущенных позитивов — ×5!), **популярных** (медианный ранг колонки **16** vs **46** — ×3) и чуть более свежих (2.3 vs 5.2 дня). То есть additive-накопление **перевешивает «привычные»/популярные пары** и заслоняет ими сегодняшние реальные интересы — это и есть сатурация массы времени жизни + доминирование популярных колонок, ровно то, о чём предупреждает delta-rule-теория (`adjacent_fields_dense_memory.md`).

⇒ Два главных кандидата на починку **упорядочивания**:
1. **Сатурирующая запись** — delta-rule/EMA (масса ячейки ограничена, «привычка» не растёт бесконечно) или сглаживание массы при чтении (`log1p`, степень <1);
2. **Нормализация колонок при чтении** — ранжировать по `M[u,i] / colmass_i^γ` (TF-IDF-стиль; это read-правило **того же** состояния, не второй предиктор).

Staleness-разрыв мал и знак «за» текущее κ — decay уже настроен (κ-свип это подтверждал).

In [5]:
# Эксперимент 1 — колоночная нормализация чтения: rank по M[u,i] / colmass_i^γ  (colmass — каузальный
# счётчик событий колонки в том же стриме; γ=0 — базлайн). Чинит доминирование популярных колонок.
class ColNormMemory(PairMemory):
    def __init__(self, *a, gamma=0.5, **k):
        super().__init__(*a, **k); self.gamma = float(gamma); self.colpop = np.zeros(self.C)
    def update(self, src, dst, t, w):
        super().update(src, dst, t, w)
        if src.size: np.add.at(self.colpop, dst.astype(np.int64), 1)
    def read(self, users, t_read):
        r = super().read(users, t_read)
        return r / np.maximum(self.colpop, 1.0) ** self.gamma if self.gamma > 0 else r

alpha40 = np.log(2) / (40 * DAY); base = 0.5219
print(f"базлайн (γ=0): 0.5219")
exp1 = {}
for g in [0.25, 0.5, 0.75, 1.0]:
    nd = eval_ranker(DS, ColNormMemory(N, C, alpha40, rphi, write="add", gamma=g))
    exp1[g] = nd; print(f"γ={g:.2f}: NDCG@10 = {nd:.4f}   ({nd-base:+.4f})")

базлайн (γ=0): 0.5219


γ=0.25: NDCG@10 = 0.4910   (-0.0309)


γ=0.50: NDCG@10 = 0.3830   (-0.1389)


γ=0.75: NDCG@10 = 0.2361   (-0.2858)


γ=1.00: NDCG@10 = 0.1373   (-0.3846)


**Вывод (эксп. 1): колоночная нормализация — РЕФУТИРОВАНА** (монотонно хуже: γ=0.25 → −0.031, …, γ=1.0 → −0.385). Интерпретация важна: популярные колонки доминируют в топ-10 **не по ошибке** — сами метки скоррелированы с популярностью (юзеры реально возвращаются к популярным item'ам). Диагноз «FP популярнее FN» был **эффектом отбора**, а не признаком лечения: штраф за популярность убивает больше истинных позитивов на популярных item'ах, чем спасает хвостовых. Урок: чинить надо **массу ячейки** (×5 разрыв), а не популярность колонки (×3 разрыв был конфаундом).

In [6]:
# Эксперимент 2 — count-aware сатурация записи: m <- m*dec + phi/(1+cnt_cell)^q  (q=0 — базлайн).
# Гасит бесконечный рост «привычных» ячеек, НЕ ломая равный вклад одновременных событий.
class SatMemory(PairMemory):
    def __init__(self, *a, q=0.5, **k):
        super().__init__(*a, **k); self.q = float(q); self.cnt = np.zeros(self.M.size)  # flat per-cell counter
    def update(self, src, dst, t, w):
        if src.size == 0: return
        idx = src.astype(np.int64) * self.C + dst.astype(np.int64); tb = float(t.max())
        uniq, inv = np.unique(idx, return_inverse=True)
        sphi = np.bincount(inv, weights=self.phi(w), minlength=uniq.size)
        k = np.bincount(inv, minlength=uniq.size)
        au = self.alpha[(uniq // self.C)]
        fM, fT = self.M.reshape(-1), self.T.reshape(-1)
        dec = np.exp(-au * np.clip(tb - fT[uniq], 0, None))
        fM[uniq] = fM[uniq] * dec + sphi / (1.0 + self.cnt[uniq]) ** self.q   # сатурирующий инкремент
        fT[uniq] = tb; self.cnt[uniq] += k

exp2 = {}
for q in [0.25, 0.5, 1.0]:
    nd = eval_ranker(DS, SatMemory(N, C, alpha40, rphi, q=q))
    exp2[q] = nd; print(f"q={q:.2f}: NDCG@10 = {nd:.4f}   ({nd-base:+.4f})")

q=0.25: NDCG@10 = 0.5050   (-0.0169)


q=0.50: NDCG@10 = 0.4332   (-0.0887)


q=1.00: NDCG@10 = 0.0875   (-0.4344)


**Вывод (эксп. 2): сатурация массы — РЕФУТИРОВАНА** (q=0.25 → −0.017, q=1.0 → −0.434). Вместе с эксп. 1 это закрывает гипотезу «топ-10 забит привычными парами по ошибке»: масса тяжёлых ячеек — **настоящий сигнал** (юзеры повторяются, метки на повторах), и NDCG@10 в основном зарабатывается именно на них. FP/FN-контраст из диагноза — эффект отбора. Additive-накопление с умеренным decay уже почти оптимально «в среднем». Последняя нетестированная ось — **персонализация таймскейла**: глобальные 40 дней могут быть неверны для юзеров с сильно разным темпом активности (GLA per-row decay, ход из `adjacent_fields_dense_memory.md`, backlog T7/H5).

In [7]:
# Эксперимент 3 — per-user decay (GLA per-row): kappa_u = clip(c * mean_gap_u, 5, 365) дней,
# mean_gap_u = (span активности юзера в train) / (#его событий) — каузально, только train.
tr, _, _ = DS["data"].train_val_test_split(val_ratio=0.15, test_ratio=0.15)
ts, tt = tr.src.numpy().astype(np.int64), tr.t.numpy().astype(np.float64)
n_u = np.bincount(ts, minlength=N).astype(np.float64)
tmin = np.full(N, np.inf); tmax = np.full(N, -np.inf)
np.minimum.at(tmin, ts, tt); np.maximum.at(tmax, ts, tt)
span_d = np.where(n_u > 1, (tmax - tmin) / DAY, np.nan)
mean_gap = np.where(n_u > 1, span_d / np.maximum(n_u - 1, 1), np.nan)   # дней между событиями юзера
g = mean_gap[~np.isnan(mean_gap) & (n_u[np.arange(N)] > 1)]
print(f"mean_gap по юзерам (дней): p10={np.nanpercentile(mean_gap,10):.3f} med={np.nanmedian(mean_gap):.3f} "
      f"p90={np.nanpercentile(mean_gap,90):.3f}  → разброс темпа ×{np.nanpercentile(mean_gap,90)/np.nanpercentile(mean_gap,10):.0f}")

exp3 = {}
for c in [120, 480, 1920]:
    kappa_u = np.clip(c * np.nan_to_num(mean_gap, nan=40.0 / c * c), 5, 365)   # NaN → 40д
    kappa_u = np.where(np.isnan(mean_gap), 40.0, np.clip(c * mean_gap, 5, 365))
    alpha_u = np.log(2) / (kappa_u * DAY)
    nd = eval_ranker(DS, PairMemory(N, C, alpha_u, rphi, write="add"))
    exp3[c] = nd
    print(f"c={c:>4} (медианная κ_u={np.nanmedian(kappa_u):>5.0f}д): NDCG@10 = {nd:.4f}   ({nd-base:+.4f})")

mean_gap по юзерам (дней): p10=0.017 med=0.060 p90=0.355  → разброс темпа ×21


c= 120 (медианная κ_u=   34д): NDCG@10 = 0.5075   (-0.0144)


c= 480 (медианная κ_u=   40д): NDCG@10 = 0.5176   (-0.0043)


c=1920 (медианная κ_u=   40д): NDCG@10 = 0.5204   (-0.0015)


In [8]:
# Эксперимент 4 — label-space запись: M копит затухающие ВЕКТОРЫ МЕТОК прошлых дней (раскрытые label'ы),
# а не события-сообщения. Каузально: на дне τ читаем ДО записи меток τ. Одно состояние, одно правило.
class LabelPairMemory:
    def __init__(self, N, C, alpha):
        self.C = C; self.M = np.zeros((N, C)); self.T = np.zeros(N); self.alpha = float(alpha)
    def write_labels(self, users, labs, t):
        dec = np.exp(-self.alpha * np.clip(t - self.T[users], 0, None))
        self.M[users] = self.M[users] * dec[:, None] + labs
        self.T[users] = t
    def read(self, users, t_read):
        return self.M[users] * np.exp(-self.alpha * np.clip(t_read - self.T[users], 0, None))[:, None]

def eval_label_ranker(DS, hl_days):
    ds = DS["ds"]; mem = LabelPairMemory(N, C, np.log(2) / (hl_days * DAY)); ndcgs = []
    def stream(loader, collect):
        label_t = ds.get_label_time()
        for b in loader:
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                l0 = float(lt[0][0]); us, ys = lt[1].numpy(), lt[2].numpy(); label_t = ds.get_label_time()
                if collect:
                    pr = mem.read(us, l0)                       # читаем ДО записи сегодняшних меток
                    for i in range(len(us)):
                        if ys[i].sum() > 0: ndcgs.append(ndcg_score(ys[i:i+1], pr[i:i+1], k=10))
                mem.write_labels(us, ys, l0)                    # раскрытые метки дня пишем ПОСЛЕ
    stream(DS["loaders"]["train"], False); stream(DS["loaders"]["val"], False)
    stream(DS["loaders"]["test"], True); ds.reset_label_time()
    return float(np.mean(ndcgs))

exp4 = {}
for hl in [20, 40, 90]:
    nd = eval_label_ranker(DS, hl); exp4[hl] = nd
    print(f"label-space hl={hl:>3}д: NDCG@10 = {nd:.4f}   ({nd-base:+.4f} vs message-space 0.5219)")

label-space hl= 20д: NDCG@10 = 0.5224   (+0.0005 vs message-space 0.5219)


label-space hl= 40д: NDCG@10 = 0.5262   (+0.0043 vs message-space 0.5219)


label-space hl= 90д: NDCG@10 = 0.5258   (+0.0039 vs message-space 0.5219)


In [9]:
# Подтверждение победителя на reddit: label-space vs message-space (базлайн κ=40 → 0.5597 из p4)
DS_R = load_dataset("tgbn-reddit"); rphi_r = make_rank_phi(DS_R)
N_R, C_R = DS_R["num_nodes"], DS_R["num_classes"]
globals().update(N=N_R, C=C_R)          # eval_label_ranker использует N,C из globals — переключаем на reddit
msg_r = eval_ranker(DS_R, PairMemory(N_R, C_R, np.log(2)/(40*DAY), rphi_r, write="add"))
print(f"reddit message-space κ=40: {msg_r:.4f}  (якорь p4: 0.5597)")
lab_r = {}
for hl in [20, 40]:
    nd = eval_label_ranker(DS_R, hl); lab_r[hl] = nd
    print(f"reddit label-space hl={hl}д: {nd:.4f}   ({nd-msg_r:+.4f} vs message)")
globals().update(N=DS['num_nodes'], C=DS['num_classes'])  # вернуть genre-константы

0it [00:00, ?it/s]

80958it [00:00, 809409.02it/s]

171801it [00:00, 865311.31it/s]

264215it [00:00, 892128.58it/s]

353427it [00:00, 873397.58it/s]

...

11979875it [00:12, 939672.95it/s]

12077151it [00:12, 988420.25it/s]

reddit message-space κ=40: 0.5596  (якорь p4: 0.5597)


reddit label-space hl=20д: 0.5720   (+0.0124 vs message)


reddit label-space hl=40д: 0.5716   (+0.0120 vs message)


In [10]:
# Сводка всех проверенных правил (genre) + подтверждение на reddit
res_rows = [
    ("базлайн: message κ=40", 0.5219, "базлайн"),
    ("κ=25 (глобальный)", 0.5203, "решётка κ"),
    ("колон.норм γ=0.25", exp1[0.25], "рефутировано"), ("колон.норм γ=0.5", exp1[0.5], "рефутировано"),
    ("сатурация q=0.25", exp2[0.25], "рефутировано"), ("сатурация q=0.5", exp2[0.5], "рефутировано"),
    ("per-user κ_u (лучш.)", exp3[1920], "рефутировано"),
    ("label-space hl=20", exp4[20], "кандидат"), ("label-space hl=40", exp4[40], "ПОБЕДИТЕЛЬ"),
    ("label-space hl=90", exp4[90], "кандидат"),
]
rd = pl.DataFrame(res_rows, schema=["правило", "NDCG@10", "статус"], orient="row")
fig = px.bar(rd.to_pandas(), x="правило", y="NDCG@10", color="статус", text="NDCG@10",
             color_discrete_map={"базлайн": "#999", "решётка κ": "#bbb", "рефутировано": "#d7191c",
                                 "кандидат": "#74add1", "ПОБЕДИТЕЛЬ": "#1a9641"},
             title="CountLoader standalone (genre test NDCG@10): три рефутации и один победитель")
fig.add_hline(y=0.5219, line_dash="dash", annotation_text="базлайн 0.5219")
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.update_layout(height=470, yaxis_range=[0.35, 0.56], xaxis_tickangle=-30, showlegend=True)
fig.show()
print(f"genre : message 0.5219 → label-space 0.5262  (+0.0043)")
print(f"reddit: message 0.5596 → label-space 0.5720  (+0.0124)  — выше MovAvg(L) 0.559 из статьи!")

genre : message 0.5219 → label-space 0.5262  (+0.0043)
reddit: message 0.5596 → label-space 0.5720  (+0.0124)  — выше MovAvg(L) 0.559 из статьи!


## Вердикт

**Диагноз** («топ-10 забит тяжёлыми/популярными ячейками») породил три кандидата — все три **честно рефутированы** экспериментом: колоночная нормализация (−0.031…−0.385), сатурация массы (−0.017…−0.434), per-user κ_u (−0.0015 даже при разбросе темпа ×21). Урок: масса привычных пар и популярность колонок — **сигнал**; message-space decayed-sum с κ≈40 — локальный оптимум своего семейства.

**Победитель — смена write-target: label-space память** (копим затухающие векторы раскрытых меток прошлых дней, одно состояние, одно правило, каузально):

| | genre | reddit |
|---|---|---|
| message-space (тек. CountLoader) | 0.5219 | 0.5596 |
| **label-space hl=40д** | **0.5262** (+0.004) | **0.5720** (+0.012) |

На reddit label-space CountLoader **обходит MovAvg(L) (0.559)** — сильнейшую эвристику статьи.

**Что менять в `models/msampler.py`:** добавить label-запись — метод `write_labels(users, labels, t)` (декей строки + прибавка вектора меток; та же матрица `M`, `T` можно вести per-user) и флаг `--m_write {message,label}`. Точка интеграции в train-loop уже есть: метки раскрываются на границе дня (`get_node_label`) — писать их в `M` сразу после скоринга (read-before-write сохраняется). Селекция top-k и `soft_target` не меняются.

**Дальше:** (1) прогнать A/B сэмплера `--m_write label` vs `message` на genre (d=784, 10 эп); (2) token-проверка label-space (там coverage-gap — label-запись может помочь и покрытию); (3) осторожный вопрос за рамками: гибридная запись message+label в одну матрицу — это уже **две улики в одном состоянии** (граница no-ensemble‑гардрейла, требует явного решения владельца).